# Classification Probability Layers

Ensemble posterior、calibration、reliability、conformalの責務をprobability-spaceで整理します。


## このNotebookで学ぶこと

39番でクラス確率を出せるようになっても、その数字が信頼できるとは限りません。ここでは **予測確率の分散（ensemble）**、**確率の校正（calibration）**、**予測の信頼性（reliability）** の役割を分けて理解します。Conformalは概念だけを説明し、このNotebookでは予測集合を計算しません。

### 4つのレイヤーを混同しない

| レイヤー | 何を評価するか | このNotebookでの扱い |
|---|---|---|
| Ensemble Posterior | 複数モデル間で予測確率がどの程度異なるか | 合成メンバーと学習済みGPメンバー |
| Probability Calibration | 予測確率と観測頻度のずれ | `TemperatureScalingCalibrator` |
| Reliability / OOD | 不確かな予測や分布外入力の検知 | 概念の整理 |
| Conformal | 所定の被覆率を目指す予測集合 | 概念の整理（実装デモは対象外） |

**Active Learningは校正と別の問題です。** ラベルを取得する候補を選ぶには、目的に応じた不確実性・情報利得の指標を利用します。

In [ ]:
import torch
from botorch.sampling.get_sampler import get_sampler

from robotorchan.models.classification import (
    ClassificationEnsemblePosterior,
    TemperatureScalingCalibrator,
    classification_nll,
)

torch.set_default_dtype(torch.double)
torch.manual_seed(0)


## 1. 小さな合成データ

### 今回の合成データ

`train_X` と `train_Y` は二値分類の位置とラベルです。`grid` は図を描くための51点です。ただし、次に作るensembleは**このデータで学習した分類器から作るわけではありません**。合成した3通りの確率曲線を比較します。

In [ ]:
train_X = torch.tensor([[0.05], [0.15], [0.30], [0.48], [0.52], [0.70], [0.85], [0.95]])
train_Y = (train_X.squeeze(-1) > 0.5).to(dtype=torch.double)
grid = torch.linspace(0.0, 1.0, 51).unsqueeze(-1)


## 2. Probability ensemble posterior

### 3つの確率曲線を合成する

`p`、`p**0.8`、`p**1.2` の3通りでクラス1確率を作っています。どのメンバーでも、クラス0確率とクラス1確率の合計は1です。

`posterior.mean` はメンバー平均、`posterior.variance` はメンバー間の確率のばらつきを表します。**観測ラベルの分散** と同一ではありません。

In [ ]:
p = grid.squeeze(-1)
members = torch.stack([
    torch.stack((1.0 - p, p), dim=-1),
    torch.stack((1.0 - p.pow(0.8), p.pow(0.8)), dim=-1),
    torch.stack((1.0 - p.pow(1.2), p.pow(1.2)), dim=-1),
])
posterior = ClassificationEnsemblePosterior(members)
print(posterior.mean.shape)
print(posterior.variance.shape)
assert posterior.mean.shape == (51, 2)


### EnsembleもBoTorch-nativeな確率Posterior

`ClassificationEnsemblePosterior` はBoTorch `EnsemblePosterior` を継承します。
`mean` / `variance` は**メンバー間**の確率の平均・分散です。
`credible_interval(0.95)` は離散的なメンバー確率を重み付きでソートして求める**周辺経験区間**で、
同時95%被覆、予測ラベルの区間、各GP内部の潜在不確実性を保証しません。

`get_sampler` はBoTorchの `IndexSampler` を使い、各サンプルで**メンバーを丸ごと選択**します。
`q>1` の候補に対しても、候補ごとに別メンバーを引くのではなく、同じメンバーを参照します。


In [ ]:
ensemble_ci_lower, ensemble_ci_upper = posterior.credible_interval(0.95)
sampler = get_sampler(posterior, sample_shape=torch.Size([64]), seed=7)
ensemble_samples = sampler(posterior)
print("95% interval:", ensemble_ci_lower.shape, ensemble_ci_upper.shape)
print("BoTorch sampler:", type(sampler).__name__, ensemble_samples.shape)
assert ensemble_ci_lower.shape == ensemble_ci_upper.shape == posterior.mean.shape
assert ensemble_samples.shape == (64, 51, 2)
torch.testing.assert_close(
    ensemble_samples.sum(dim=-1), torch.ones_like(ensemble_samples[..., 0])
)


### Entropyとメンバー間の情報不一致

`posterior.predictive_entropy` は**アンサンブル平均確率のEntropy**です。一方、`posterior.expected_class_entropy` は各メンバーのEntropyを平均した値です。

両者の差 `posterior.mutual_information` は、メンバー間の予測の食い違いに由来する情報量（BALD-style）を表します。`posterior.variance` と同じ量ではありません。次のセルで比較します。

In [ ]:
predictive_entropy = posterior.predictive_entropy
expected_entropy = posterior.expected_class_entropy
ensemble_information = posterior.mutual_information
assert predictive_entropy.shape == expected_entropy.shape == ensemble_information.shape == (51,)
torch.testing.assert_close(
    ensemble_information,
    (predictive_entropy - expected_entropy).clamp_min(0.0),
)
print("maximum member disagreement:", posterior.variance[..., 1].max().item())
print("maximum BALD-style information:", ensemble_information.max().item())

## 3. Probability calibration

### Temperature scalingの読み方

`raw` は各行が `[P(y=0), P(y=1)]` の確率、`targets` は正解ラベルです。`calibrator.fit(raw, targets)` は温度パラメータを推定し、`calibrator(raw)` が補正後の確率を返します。

この4点は校正にも評価にも使われているため、表示されるNLLの改善は**汎化性能を証明しません**。実運用では校正用と最終評価用のデータを分けます。

In [ ]:
raw = torch.tensor([[0.90, 0.10], [0.65, 0.35], [0.30, 0.70], [0.05, 0.95]])
targets = torch.tensor([0, 0, 1, 1])
calibrator = TemperatureScalingCalibrator()
calibrator.fit(raw, targets)
calibrated = calibrator(raw)
print(classification_nll(raw, targets).item())
print(classification_nll(calibrated, targets).item())


### 温度スケーリングの注意

温度スケーリングは確率ベクトルの対数を温度で割り、softmaxで再正規化します。温度は校正用データの負の対数尤度（NLL）を最小化して求めます。

この例は4件のみで、校正と評価に同じデータを使っています。**校正前後のNLL改善が独立データで再現するとは限りません。** 実務では学習用・校正用・最終評価用のデータを分離します。

## 4. Reliability / conformalとの違い

Reliability/OODはentropy・disagreement・input distanceを分離して診断します。Split conformalはheld-out calibration setからprediction setを構成します。probability calibrationとは別レイヤーです。

実運用ではmodel fitting、probability calibration、conformal calibrationに使うデータの役割を明示してください。

## 5. Probability-space候補選択とラベル評価

このNotebookのensembleは合成確率メンバーであり、学習済み分類器ではありません。Probability calibrationは候補最適化ではなく確率の補正です。ここではensemble disagreementによる候補選択を示します。


### 次の候補はどこか

`posterior.variance[..., 1]` はクラス1確率のメンバー間分散です。その最大点を `grid` から選びます。これは学習済みGPから獲得関数を最大化した結果ではなく、説明用の有限候補選択です。

In [ ]:
candidate_disagreement = posterior.variance[..., 1]
candidate = grid[candidate_disagreement.argmax()].unsqueeze(0)
observed_label = (candidate.squeeze(-1) > 0.5).long()
print("candidate:", candidate)
print("observed label:", observed_label)


## 6. 図で読む：予測平均、メンバー間分散、確率校正

左の図では3つの合成メンバーと平均予測、加重経験分布に基づく95%区間を比較します。右の図はメンバー間の分散です。校正前後の4点も別図で比較します。これらは概念の可視化であり、モデルの実測精度評価ではありません。

In [ ]:
import matplotlib.pyplot as plt

xs = grid.squeeze(-1).detach().cpu().numpy()
member_values = members[..., 1].detach().cpu().numpy()
mean_values = posterior.mean[..., 1].detach().cpu().numpy()
variance_values = candidate_disagreement.detach().cpu().numpy()
ci_lower_values = ensemble_ci_lower[..., 1].detach().cpu().numpy()
ci_upper_values = ensemble_ci_upper[..., 1].detach().cpu().numpy()
selected_x = candidate.detach().cpu().reshape(-1)[0].item()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for idx, values in enumerate(member_values):
    axes[0].plot(xs, values, alpha=0.65, label=f"Member {idx + 1}")
axes[0].plot(xs, mean_values, linewidth=2, label="Ensemble mean")
axes[0].fill_between(
    xs, ci_lower_values, ci_upper_values,
    alpha=0.2, label="95% weighted member interval",
)
axes[0].set_ylabel("P(y=1)")
axes[0].set_xlabel("Input x")
axes[0].legend()
axes[1].plot(xs, variance_values, label="Member disagreement")
axes[1].axvline(selected_x, linestyle="--", color="gray", label="Candidate")
axes[1].set_ylabel("Variance of P(y=1)")
axes[1].set_xlabel("Input x")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
indices = torch.arange(raw.shape[0]).cpu().numpy()
ax.plot(indices, raw[:, 1].detach().cpu().numpy(), "o-", label="Before")
ax.plot(indices, calibrated[:, 1].detach().cpu().numpy(), "s--", label="After")
ax.scatter(indices, targets.detach().cpu().numpy(), marker="x", label="True label")
ax.set_xticks(indices)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel("Sample index")
ax.set_ylabel("P(y=1) / label")
ax.set_title("Temperature scaling on demonstration data")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### 読み取りと評価の注意

- モデル間の不一致が大きい場所と、平均予測が0.5付近の場所は同一とは限りません。
- 校正後の確率が大きく変わっても、それだけで分類性能が改善したとはいえません。
- 実際の校正性能は、独立した評価データに対してNLL・Brier Score・reliability diagram等で確認します。

## 7. 学習済みGP EnsembleでActive Learning候補を提案

ここまでは合成した確率曲線の説明でした。ここからは実際に2つの `BinarySingleTaskGPClassifier` を公開 `fit()` APIで学習し、`GPBinaryClassificationEnsemble` にまとめます。

モデルの学習データは同一ですが、異なる乱数シードで初期化することで、有限学習の下で異なる予測が生じる可能性があります。メンバー間不一致が大きい候補を選び、合成ラベルを取得して再学習します。

**注意：** 2モデル・少数epochのデモです。Ensemble分散や情報量が十分に校正されているとは限りません。

In [ ]:
from robotorchan.models.classification import (
    BinarySingleTaskGPClassifier,
    GPBinaryClassificationEnsemble,
)

trained_members = []
for seed in (11, 23):
    torch.manual_seed(seed)
    member = BinarySingleTaskGPClassifier(train_X, train_Y, inducing_points=6)
    member.fit(epochs=30, lr=0.05)
    member.eval()
    trained_members.append(member)

trained_ensemble = GPBinaryClassificationEnsemble(*trained_members)
with torch.no_grad():
    trained_posterior = trained_ensemble.probability_posterior(grid)
    trained_probability = trained_posterior.mean[..., 1]
    trained_information = trained_posterior.mutual_information
    trained_disagreement = trained_posterior.variance[..., 1]

print("trained ensemble probability:", trained_probability.shape)
print("BALD-style information:", trained_information.shape)
assert trained_probability.shape == trained_information.shape == (51,)

### 学習済みモデルによる候補提案

次にラベルを取得する候補は、学習済みEnsembleの `mutual_information` を**有限候補プールで最大化**して選びます。これは単純な平均クラス1確率の最大化ではなく、モデル間の予測の食い違いを重視するActive Learningです。

今回はPosteriorの公開APIを直接評価します。BoTorchの `optimize_acqf` に渡す専用獲得関数を新規実装したわけではありません。

In [ ]:
trained_candidate_idx = trained_information.argmax()
trained_candidate = grid[trained_candidate_idx].reshape(1, 1)
trained_label = (trained_candidate[:, 0] > 0.5).to(dtype=train_Y.dtype)
print("proposed x:", trained_candidate.item())
print("observed label:", int(trained_label.item()))

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].plot(xs, trained_probability.detach().cpu().numpy(), label="ensemble P(y=1)")
axes[0].scatter(train_X[:, 0], train_Y, label="training labels")
axes[0].set(ylabel="P(y=1)", title="Fitted GP classification ensemble")
axes[0].legend()
axes[1].plot(xs, trained_information.detach().cpu().numpy(), label="BALD-style information")
axes[1].plot(xs, trained_disagreement.detach().cpu().numpy(), label="member variance")
axes[1].axvline(trained_candidate.item(), linestyle="--", color="gray", label="candidate")
axes[1].set(xlabel="x", ylabel="acquisition score")
axes[1].legend()
plt.tight_layout()
plt.show()

### ラベル追加と再学習

取得したラベルを履歴に追加して2モデルを再学習します。学習済みEnsembleをその場で更新するのではなく、新しい学習データから構築し直します。

In [ ]:
updated_train_X = torch.cat((train_X, trained_candidate.detach()), dim=0)
updated_train_Y = torch.cat((train_Y, trained_label), dim=0)
assert updated_train_X.shape == (9, 1)
assert updated_train_Y.shape == (9,)

updated_members = []
for seed in (11, 23):
    torch.manual_seed(seed)
    member = BinarySingleTaskGPClassifier(updated_train_X, updated_train_Y, inducing_points=6)
    member.fit(epochs=30, lr=0.05)
    member.eval()
    updated_members.append(member)
updated_ensemble = GPBinaryClassificationEnsemble(*updated_members)
with torch.no_grad():
    updated_probability = updated_ensemble.predict_proba(grid)[..., 1]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(xs, trained_probability.detach().cpu().numpy(), label="before")
ax.plot(xs, updated_probability.detach().cpu().numpy(), label="after")
ax.axvline(trained_candidate.item(), linestyle="--", color="gray", label="new label")
ax.set(xlabel="x", ylabel="P(y=1)", title="Ensemble probability before / after labeling")
ax.legend()
plt.tight_layout()
plt.show()

## 初学者向け補足：確率を出した後にも評価が必要

分類器が `0.9` と予測しても、実際に同じような予測を100件集めて約90件がクラス1になるとは限りません。確率の数値と実際の頻度が一致する性質を **calibration（確率校正）** と呼びます。

| 概念 | 答えたい問い |
|---|---|
| Ensemble | 複数モデルの予測はどの程度一致するか |
| Calibration | 予測確率と実際の頻度は整合しているか |
| Reliability / OOD | 未知の入力や信頼しづらい予測を検出できるか |
| Conformal prediction | 仮定のもとで所定の被覆率を目指す予測集合を作れるか |

### このNotebookで注意すること

このNotebookのensembleは **合成した確率メンバー** を用いる説明例です。学習済みGP分類器を複数訓練した性能実験ではありません。

校正を評価するときは、学習・校正・最終評価のデータを適切に分離します。Brier ScoreやLog Lossは確率の質を見る指標ですが、Accuracyとは測っているものが異なります。Conformalの被覆率も、必要な交換可能性などの仮定を満たすか確認してください。

### 読み取りの練習

予測確率が0.5に近い点と、メンバー間の意見が割れる点は必ずしも同じではありません。各セルの出力を見ながら、どの指標がどちらを強調するか考えてみてください。

## まとめ：確率の信頼性とActive Learning

| 目的 | 使用API |
|---|---|
| 学習済み二値分類器 | `BinarySingleTaskGPClassifier.fit()` |
| Ensemble | `GPBinaryClassificationEnsemble` |
| 確率Posterior | `ensemble.probability_posterior()` |
| メンバー間不一致 | `posterior.variance` / `posterior.mutual_information` |
| 確率校正 | `TemperatureScalingCalibrator.fit()` |
| 候補提案 | 学習済みEnsembleの情報量を有限候補で最大化 |
| 観測更新 | 新ラベルを追加して各メンバーを再学習 |

**Ensembleの不一致、予測ラベルの曖昧さ、確率校正、Conformalの被覆率は別の概念です。** 目的に応じて正しい確率・不確実性を選択します。